# Migrar faltantes de SharePoint al volumen

Copia (nunca mueve) solo los archivos marcados como "Candidato a migrar" o "Revisar" en `Cruce_SharePoint_vs_Volumen.xlsx`, desde la carpeta sincronizada de OneDrive hacia `Y:\Galería\FOTOS`, preservando la misma subcarpeta de autor/especie que tenían en SharePoint.

**No copia la carpeta `FOTOS` completa.** Solo los ~622 archivos que el cruce ya identificó como ausentes o dudosos. El resto ya está en el volumen.

**Por qué copiar y no mover:** la carpeta sincronizada de OneDrive es de dos vías con SharePoint real. Mover o borrar ahí afecta el sitio institucional, no solo tu copia local.

**Los casos "Revisar: nombre coincide, tamaño distinto"** se copian aparte, a una subcarpeta `_revisar_manualmente`, en vez de mezclarlos directo en la estructura de autor/especie, porque no sabemos con certeza si es una edición legítima o un archivo distinto.

In [ ]:
import os
import shutil
from pathlib import Path
from datetime import datetime

import pandas as pd

# <<< AJUSTA ESTA RUTA a la que te dio "Copiar como ruta de acceso" en tu carpeta FOTOS sincronizada
import json

def cargar_config(archivo="../config_local.json"):
    ruta = Path(archivo)
    if not ruta.exists():
        raise FileNotFoundError(
            f"No se encontro {archivo}. Copia config_local.example.json a {archivo} "
            "(junto a este notebook) y ajusta tus rutas locales. No se sube a git."
        )
    return json.loads(ruta.read_text(encoding="utf-8"))

config = cargar_config()
RAIZ_SINCRONIZADA = Path(config["raiz_sincronizada_documentos"])

RAIZ_GALERIA = Path(config["raiz_galeria"])
DESTINO_FOTOS = RAIZ_GALERIA / "FOTOS"
DESTINO_REVISAR = RAIZ_GALERIA / "_revisar_manualmente"

RUTA_CRUCE = Path("../salida") / "Cruce_SharePoint_vs_Volumen.xlsx"
CARPETA_SALIDA = Path("../salida")
CARPETA_SALIDA.mkdir(exist_ok=True)

# Dos prefijos DISTINTOS, no confundirlos:
# PREFIJO_FOTOS: siempre fijo, es la parte de ruta_completa que hay que quitar para saber
#   dónde cae el archivo DENTRO de Galería\FOTOS. No depende de cómo sincronizaste.
# PREFIJO_SINCRONIZADO: depende de qué sincronizaste. Vacío ("") si sincronizaste la
#   biblioteca completa de Documentos (la ruta local ya incluye "Bases de datos\FOTOS\").
#   "Bases de datos/FOTOS/" si sincronizaste esa subcarpeta directo.
PREFIJO_FOTOS = "Bases de datos/FOTOS/"
PREFIJO_SINCRONIZADO = config["prefijo_relativo_fotos"]

print(f"Raíz sincronizada: {RAIZ_SINCRONIZADA} ({'existe' if RAIZ_SINCRONIZADA.exists() else 'NO ENCONTRADA, ajusta la ruta arriba'})")

In [ ]:
df_pendientes = pd.read_excel(RUTA_CRUCE, sheet_name="Por migrar o revisar")
print(f"Filas a procesar: {len(df_pendientes):,}")
print(df_pendientes["estado_cruce"].value_counts())

def calcular_rutas(row):
    ruta_completa = row["ruta_completa"]
    if not ruta_completa.startswith(PREFIJO_FOTOS):
        return pd.Series({"ruta_relativa": None, "origen_local": None, "destino": None})
    rel_fotos = ruta_completa[len(PREFIJO_FOTOS):]  # para el destino en Galería

    if PREFIJO_SINCRONIZADO and ruta_completa.startswith(PREFIJO_SINCRONIZADO):
        rel_sync = ruta_completa[len(PREFIJO_SINCRONIZADO):]
    else:
        rel_sync = ruta_completa  # sincronizaste la biblioteca completa, no se quita nada

    origen = RAIZ_SINCRONIZADA / rel_sync
    destino = (DESTINO_REVISAR if row["estado_cruce"].startswith("Revisar") else DESTINO_FOTOS) / rel_fotos
    return pd.Series({"ruta_relativa": rel_fotos, "origen_local": str(origen), "destino": str(destino)})

df_pendientes = pd.concat([df_pendientes, df_pendientes.apply(calcular_rutas, axis=1)], axis=1)

sin_prefijo = df_pendientes["ruta_relativa"].isna().sum()
if sin_prefijo:
    print(f"AVISO: {sin_prefijo} filas no calzaron con el prefijo esperado, revisar antes de continuar.")

# Verificación rápida: si ni siquiera una muestra existe en la carpeta sincronizada,
# probablemente OneDrive todavía no termina de sincronizar. Mejor avisar aquí que
# descubrirlo hasta el final con 622 fallos.
muestra = df_pendientes["origen_local"].dropna().sample(min(15, len(df_pendientes)), random_state=1)
existen = muestra.apply(lambda p: Path(p).exists()).sum()
print(f"\nVerificación de la carpeta sincronizada: {existen} de {len(muestra)} archivos de muestra sí existen localmente.")
if existen == 0:
    print("NINGUNO existe todavia. Revisa el icono de OneDrive: si dice \"Sincronizando\", espera a que termine antes de activar la copia real.")

df_pendientes.to_excel(CARPETA_SALIDA / "Plan_Migracion_SharePoint.xlsx", index=False)
print(f"Plan guardado: {CARPETA_SALIDA / 'Plan_Migracion_SharePoint.xlsx'}")

## Ejecución real

Revisa `Plan_Migracion_SharePoint.xlsx` antes de activar esto.

In [ ]:
EJECUTAR_COPIA_REAL = False  # <-- cámbialo a True solo después de revisar el plan

if not EJECUTAR_COPIA_REAL:
    print("EJECUTAR_COPIA_REAL está en False. No se copió nada. Revisa el plan y vuelve aquí.")
else:
    log = []
    exitosos = fallidos = omitidos = 0

    for _, row in df_pendientes.iterrows():
        if not row["origen_local"] or not row["destino"]:
            log.append((row["ruta_completa"], "", "omitido", "sin ruta relativa calculable"))
            omitidos += 1
            continue

        origen, destino = Path(row["origen_local"]), Path(row["destino"])
        try:
            if destino.exists():
                log.append((str(origen), str(destino), "omitido", "el destino ya existe"))
                omitidos += 1
                continue
            if not origen.exists():
                raise FileNotFoundError(f"No existe en la carpeta sincronizada: {origen}")

            destino.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(str(origen), str(destino))  # copy2 preserva fechas; copy, no move
            log.append((str(origen), str(destino), "ok", ""))
            exitosos += 1
        except Exception as e:
            log.append((str(origen), str(destino), "error", str(e)))
            fallidos += 1

    df_log = pd.DataFrame(log, columns=["origen", "destino", "resultado", "detalle"])
    ruta_log = CARPETA_SALIDA / f"Log_Migracion_SharePoint_{datetime.now():%Y%m%d_%H%M%S}.csv"
    df_log.to_csv(ruta_log, index=False, encoding="utf-8")

    print(f"Copiados: {exitosos:,}  Omitidos (ya existían o sin ruta): {omitidos:,}  Fallidos: {fallidos:,}")
    print(f"Log guardado: {ruta_log}")
    print(f"\nSiguiente paso: vuelve a correr Volumen_Fotos_BD.ipynb para que el escaneo agarre estos archivos nuevos.")